# Laboratorium 1: Sieci semantyczne i ontologie

Dziedzina: mała ontologia uczelni (przestrzeń nazw `http://example.org/uczelnia#`, prefiks `u:`) oraz Wikidata. Część modelowania odbywa się w WebProtégé, a eksport stamtąd sprawdzamy tutaj. Cel: przejść od sieci semantycznej z dziedziczeniem do grafu RDF, zapytań SPARQL i wnioskowania w RDFS i OWL.

In [ ]:
# rdflib, owlrl i owlready2 nie są preinstalowane w Colabie
%pip install -q rdflib owlrl owlready2

In [ ]:
import io
import shutil
import subprocess
import time
import zipfile
from collections import Counter, deque
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import requests

import owlrl
import rdflib
from rdflib import OWL, RDF, RDFS, XSD, Graph, Literal, Namespace, URIRef

import owlready2
from owlready2 import (AllDifferent, AllDisjoint, OwlReadyInconsistentOntologyError, TransitiveProperty,
                       World, close_world, sync_reasoner_hermit)

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 80)
U = Namespace("http://example.org/uczelnia#")
print("rdflib", rdflib.__version__, "| owlrl", owlrl.__version__, "| owlready2", owlready2.VERSION,
      "| networkx", nx.__version__)

## Zadanie 1: Sieć semantyczna i dziedziczenie

Sieć poniżej jest gotowa. Każdy węzeł ma słownik `cechy` z własnościami zapisanymi wprost, krawędzie mają atrybut `rel` (`is-a` albo `part-of`).

In [ ]:
siec = nx.DiGraph()

def dodaj(wezel, **cechy):
    siec.add_node(wezel, cechy=cechy)

dodaj("zwierzę", oddycha=True, je=True)
dodaj("ptak", ma_piora=True, lata=True, liczba_nog=2)
dodaj("ryba", ma_skrzela=True, plywa=True, lata=False)
dodaj("kanarek", spiewa=True, kolor="żółty")
dodaj("struś", lata=False, kolor="szary")
dodaj("pingwin", lata=False, plywa=True)
dodaj("rekin", grozny=True)
dodaj("łosoś", jadalny=True, kolor="srebrny")
dodaj("Tweety")
dodaj("Pingu")
dodaj("skrzydło", sluzy_do="lotu")

for dziecko, rodzic in [("ptak", "zwierzę"), ("ryba", "zwierzę"), ("kanarek", "ptak"),
                        ("struś", "ptak"), ("pingwin", "ptak"), ("rekin", "ryba"),
                        ("łosoś", "ryba"), ("Tweety", "kanarek"), ("Pingu", "pingwin")]:
    siec.add_edge(dziecko, rodzic, rel="is-a")
siec.add_edge("skrzydło", "ptak", rel="part-of")
print(siec.number_of_nodes(), "węzłów,", siec.number_of_edges(), "krawędzi")

In [ ]:
# Gotowy rysunek sieci
# poziom węzła = odległość od korzenia, potrzebna tylko do rysunku
poziom = nx.shortest_path_length(siec.to_undirected(), "zwierzę")
nx.set_node_attributes(siec, poziom, "poziom")
pos = nx.multipartite_layout(siec, subset_key="poziom", align="horizontal")
pos = {n: (x, -y) for n, (x, y) in pos.items()}

fig, ax = plt.subplots(figsize=(11, 5))
kolory = ["#f5d5d5" if n in ("Tweety", "Pingu") else "#dde8f5" for n in siec]
nx.draw_networkx(siec, pos, ax=ax, node_color=kolory, node_size=1700, font_size=9,
                 edgecolors="#33507a", arrowsize=15)
nx.draw_networkx_edge_labels(siec, pos, ax=ax, font_size=8,
                             edge_labels=nx.get_edge_attributes(siec, "rel"))
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Napisz znajdz_ceche(siec, wezel, cecha): przeszukiwanie wszerz w górę po krawędziach is-a,
# zwraca listę (wartość, źródło, odległość) dla najbliższych węzłów z cechą zapisaną wprost.
# Napisz odpowiedz(...), która zwraca wartość, "konflikt" albo "nie wiadomo".
# Zbuduj tabelę odpowiedzi dla par (węzeł, cecha) z instrukcji.

In [ ]:
# Policz cechy zapisane wprost i pary (węzeł, cecha), na które sieć umie odpowiedzieć

In [ ]:
# Zbuduj diament Nixona i sprawdź odpowiedź dla cechy "pacyfista".
# Na kopii sieci dodaj krawędź Pingu is-a ptak i sprawdź, co sieć odpowiada na pytanie, czy Pingu lata.

In [ ]:
# Popraw rozstrzyganie konfliktów: zamiast najkrótszej ścieżki odrzuć trafienia z węzłów,
# pod którymi w hierarchii leży inne trafienie. Sprawdź Pingu i Nixona.

### Wnioski do zadania 1

(uzupełnij)

## Zadanie 2: RDF i Turtle w rdflib

Dane uczelni w składni Turtle (sama warstwa faktów, bez schematu). Komórka jest gotowa.

In [ ]:
UCZELNIA_TTL = """
@prefix u:    <http://example.org/uczelnia#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

# ---------- wydziały ----------
u:WI a u:Wydzial ; rdfs:label "Wydział Informatyki"@pl .
u:WZ a u:Wydzial ; rdfs:label "Wydział Zarządzania"@pl .

# ---------- kursy ----------
u:Programowanie a u:KursObowiazkowy ; rdfs:label "Programowanie"@pl ; u:ects 6 ; u:naWydziale u:WI .
u:Algorytmy a u:KursObowiazkowy ; rdfs:label "Algorytmy i struktury danych"@pl ; u:ects 6 ;
    u:naWydziale u:WI ; u:wymagaKursu u:Programowanie .
u:Logika a u:KursObowiazkowy ; rdfs:label "Logika dla informatyków"@pl ; u:ects 4 ; u:naWydziale u:WI .
u:SztucznaInteligencja a u:KursObowiazkowy ; rdfs:label "Sztuczna inteligencja"@pl ; u:ects 5 ;
    u:naWydziale u:WI ; u:wymagaKursu u:Algorytmy , u:Logika .
u:ReprezentacjaWiedzy a u:KursObieralny ; rdfs:label "Metody reprezentacji wiedzy"@pl ; u:ects 4 ;
    u:naWydziale u:WI ; u:wymagaKursu u:SztucznaInteligencja .
u:Statystyka a u:KursObowiazkowy ; rdfs:label "Statystyka"@pl ; u:ects 5 ; u:naWydziale u:WZ .
u:UczenieMaszynowe a u:KursObieralny ; rdfs:label "Uczenie maszynowe"@pl ; u:ects 5 ;
    u:naWydziale u:WI ; u:wymagaKursu u:Statystyka , u:Programowanie .
u:Marketing a u:KursObieralny ; rdfs:label "Marketing internetowy"@pl ; u:ects 3 ; u:naWydziale u:WZ .

# ---------- pracownicy ----------
u:p01 a u:Pracownik ; u:imie "Anna" ; u:nazwisko "Kowalska" ; u:rokUrodzenia 1978 ;
    u:naWydziale u:WI ; u:prowadzi u:SztucznaInteligencja , u:ReprezentacjaWiedzy .
u:p02 a u:Pracownik ; u:imie "Piotr" ; u:nazwisko "Nowak" ; u:rokUrodzenia 1985 ;
    u:naWydziale u:WI ; u:prowadzi u:Programowanie , u:Algorytmy .
u:p03 a u:Pracownik ; u:imie "Krzysztof" ; u:nazwisko "Lewandowski" ; u:rokUrodzenia 1969 ;
    u:naWydziale u:WZ ; u:prowadzi u:Statystyka .
u:p04 a u:Pracownik ; u:imie "Barbara" ; u:nazwisko "Wójcik" ; u:rokUrodzenia 1990 ;
    u:naWydziale u:WZ .
u:p05 u:imie "Tomasz" ; u:nazwisko "Kamiński" ; u:rokUrodzenia 1982 ;
    u:prowadzi u:UczenieMaszynowe , u:Logika .
u:p06 a u:Pracownik ; u:imie "Ewa" ; u:nazwisko "Zielińska" ; u:rokUrodzenia 1975 ;
    u:naWydziale u:WZ ; u:prowadzi u:Marketing .

# ---------- studenci ----------
u:s01 a u:Student ; u:imie "Jan" ; u:nazwisko "Wiśniewski" ; u:rokStudiow 1 ; u:rokUrodzenia 2001 ;
    u:zaliczyl u:Programowanie , u:Algorytmy , u:Logika , u:SztucznaInteligencja ;
    u:zapisanyNa u:ReprezentacjaWiedzy .
u:s02 a u:Student ; u:imie "Katarzyna" ; u:nazwisko "Dąbrowska" ; u:rokStudiow 2 ; u:rokUrodzenia 2000 ;
    u:zaliczyl u:Programowanie , u:Algorytmy , u:Logika , u:SztucznaInteligencja , u:Statystyka ;
    u:zapisanyNa u:ReprezentacjaWiedzy , u:UczenieMaszynowe .
u:s03 a u:Student ; u:imie "Michał" ; u:nazwisko "Kozłowski" ; u:rokStudiow 1 ; u:rokUrodzenia 2002 ;
    u:zaliczyl u:Programowanie , u:Statystyka ;
    u:zapisanyNa u:UczenieMaszynowe , u:Algorytmy .
u:s04 a u:Student ; u:imie "Agnieszka" ; u:nazwisko "Jankowska" ; u:rokStudiow 1 ; u:rokUrodzenia 2001 ;
    u:zaliczyl u:Programowanie ;
    u:zapisanyNa u:ReprezentacjaWiedzy .
u:s05 a u:Student ; u:imie "Paweł" ; u:nazwisko "Mazur" ; u:rokStudiow 2 ; u:rokUrodzenia 1999 ;
    u:zaliczyl u:Programowanie , u:Algorytmy , u:Logika , u:SztucznaInteligencja , u:ReprezentacjaWiedzy , u:Statystyka , u:UczenieMaszynowe .
u:s06 a u:Student ; u:imie "Magdalena" ; u:nazwisko "Krawczyk" ; u:rokStudiow 1 ; u:rokUrodzenia 2002 ;
    u:zaliczyl u:Statystyka ; u:zapisanyNa u:Marketing .
u:s07 a u:Student ; u:imie "Marcin" ; u:nazwisko "Piotrowski" ; u:rokStudiow 2 ; u:rokUrodzenia 2000 ;
    u:zaliczyl u:Statystyka , u:Marketing .
u:s08 a u:Student ; u:imie "Joanna" ; u:nazwisko "Grabowska" ; u:rokStudiow 1 ; u:rokUrodzenia 2001 ;
    u:zapisanyNa u:Programowanie , u:Logika , u:Statystyka .
u:s09 a u:Student ; u:imie "Łukasz" ; u:nazwisko "Pawłowski" ; u:rokStudiow 2 ; u:rokUrodzenia 1998 ;
    u:zaliczyl u:Programowanie , u:Algorytmy , u:Logika ;
    u:zapisanyNa u:SztucznaInteligencja .
u:s10 u:imie "Aleksandra" ; u:nazwisko "Michalska" ; u:rokUrodzenia 2002 ;
    u:zapisanyNa u:Marketing , u:Statystyka .
u:s11 a u:Student ; u:imie "Kamil" ; u:nazwisko "Zając" ; u:rokStudiow 1 ; u:rokUrodzenia 2002 ;
    u:zaliczyl u:Programowanie , u:Logika ;
    u:zapisanyNa u:Algorytmy .
u:s12 a u:Student ; u:imie "Natalia" ; u:nazwisko "Król" ; u:rokStudiow 2 ; u:rokUrodzenia 2000 ;
    u:zaliczyl u:Programowanie , u:Algorytmy , u:Logika , u:SztucznaInteligencja .

# ---------- prace dyplomowe ----------
u:pd01 a u:PracaDyplomowa ; rdfs:label "Ontologia planu studiów w OWL"@pl ;
    u:autor u:s05 ; u:promotor u:p01 .
u:pd02 a u:PracaDyplomowa ; rdfs:label "Rekomendacja kursów obieralnych"@pl ;
    u:autor u:s02 ; u:promotor u:p02 .
u:pd03 a u:PracaDyplomowa ; rdfs:label "Analiza ankiet studenckich"@pl ;
    u:autor u:s07 ; u:promotor u:p04 .
"""

In [ ]:
PREFIKSY = """
PREFIX u:    <http://example.org/uczelnia#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
"""
SKROTY = [("u:", str(U)), ("rdf:", str(RDF)), ("rdfs:", str(RDFS)), ("owl:", str(OWL)), ("xsd:", str(XSD))]

def krotko(x):
    # IRI skracamy do postaci z prefiksem, literał zamieniamy na wartość Pythona
    if x is None:
        return None
    if isinstance(x, Literal):
        return x.toPython()
    s = str(x)
    for pref, ns in SKROTY:
        if s.startswith(ns):
            return pref + s[len(ns):]
    return s

def tabela(graf, zapytanie):
    wynik = graf.query(PREFIKSY + zapytanie)
    return pd.DataFrame([[krotko(v) for v in w] for w in wynik], columns=[str(v) for v in wynik.vars])

In [ ]:
# Wczytaj UCZELNIA_TTL do grafu rdflib, powiąż prefiks u:, wypisz liczbę trójek, liczbę różnych podmiotów,
# liczbę trójek z literałem w obiekcie i tabelę liczby trójek dla każdego predykatu

In [ ]:
# Ile zasobów ma każdy typ (rdf:type)? Które osoby nie mają żadnego typu?

In [ ]:
# Wypisz osoby (zasoby z u:imie), które nie mają rdf:type.
# Sprawdź, czy w grafie jest trójka (u:p01, u:imie, Literal("Anna")) oraz ta sama z Literal("Anna", lang="pl")

In [ ]:
# Dodaj programowo studentkę u:s13 (Zofia Wróbel, rok studiów 1, urodzona 2003, zapisana na
# u:Programowanie i u:Logika). Zserializuj jej trójki do Turtle i N-Triples.
# Porównaj długość serializacji całego grafu w obu formatach.

In [ ]:
# Gotowa komórka: prerekwizyty i prowadzący jako graf networkx
G = nx.DiGraph()
for s, p, o in graf:
    if p in (U.wymagaKursu, U.prowadzi):
        G.add_edge(krotko(s)[2:], krotko(o)[2:], rel=krotko(p))
print("acykliczny:", nx.is_directed_acyclic_graph(G))
wymaga = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["rel"] == "u:wymagaKursu"])
print("najdłuższy łańcuch prerekwizytów:", nx.dag_longest_path(wymaga))

kolor = {n: ("#f5e1c8" if (U[n], RDF.type, U.KursObowiazkowy) in graf else
             "#d9ecd0" if (U[n], RDF.type, U.KursObieralny) in graf else "#dde8f5") for n in G}
pos = nx.spring_layout(G, seed=RANDOM_STATE, k=1.2)
fig, ax = plt.subplots(figsize=(11, 7))
nx.draw_networkx(G, pos, ax=ax, node_color=[kolor[n] for n in G], node_size=1500, font_size=8,
                 edgecolors="#555", arrowsize=14)
nx.draw_networkx_edge_labels(G, pos, ax=ax, font_size=7, edge_labels=nx.get_edge_attributes(G, "rel"))
ax.set_title("Kursy obowiązkowe (pomarańczowe), obieralne (zielone) i prowadzący (niebieskie)")
ax.axis("off")
plt.tight_layout()
plt.show()

### Wnioski do zadania 2

(uzupełnij)

## Zadanie 3: SPARQL

Funkcja `tabela(graf, zapytanie)` dokleja prefiksy i zwraca DataFrame.

In [ ]:
# 3.1 Przykład gotowy: studenci z rokiem studiów
tabela(graf, '''
SELECT ?s ?imie ?nazwisko ?rok WHERE {
  ?s a u:Student ; u:imie ?imie ; u:nazwisko ?nazwisko .
  OPTIONAL { ?s u:rokStudiow ?rok }
} ORDER BY ?nazwisko''')

In [ ]:
# 3.2 FILTER: kursy Wydziału Informatyki za co najmniej 5 ECTS; osoby urodzone przed 1980

In [ ]:
# 3.3 OPTIONAL: wszyscy pracownicy (rdf:type u:Pracownik) i nazwy prowadzonych kursów (GROUP_CONCAT),
# łącznie z pracownikami, którzy nic nie prowadzą

In [ ]:
# 3.4 GROUP BY: liczba zapisanych na każdy kurs (także kursy bez zapisanych);
# suma ECTS zaliczonych kursów na studenta, tylko studenci z co najmniej 15 ECTS (HAVING)

In [ ]:
# 3.5 Ścieżki właściwości: wszystkie (pośrednie) prerekwizyty u:ReprezentacjaWiedzy (u:wymagaKursu+);
# studenci zapisani na kurs, któremu brakuje zaliczonego prerekwizytu (FILTER NOT EXISTS)

In [ ]:
# 3.6 CONSTRUCT: zbuduj graf z trójkami ?s u:uczonyPrzez ?p (student zapisany na kurs prowadzony przez ?p).
# Ile trójek powstało? Ilu studentów uczy każdy prowadzący?

### Wnioski do zadania 3

(uzupełnij)

## Zadanie 4: RDFS i OWL 2 RL w owlrl

Schemat RDFS piszecie sami. Aksjomaty OWL są gotowe w drugiej komórce.

In [ ]:
SCHEMAT_TTL = """
@prefix u:    <http://example.org/uczelnia#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

u:Student rdfs:subClassOf u:Osoba .
# uzupełnij: pozostałe podklasy (Pracownik, Wykladowca, KursObowiazkowy, KursObieralny)
# oraz rdfs:domain i rdfs:range dla właściwości obiektowych, rdfs:domain dla u:rokStudiow i u:ects
"""
schemat = Graph().parse(data=SCHEMAT_TTL, format="turtle")
print("trójek schematu:", len(schemat))

In [ ]:
OWL_TTL = """
@prefix u:    <http://example.org/uczelnia#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .

u:wymagaKursu a owl:TransitiveProperty .
u:prowadzonyPrzez owl:inverseOf u:prowadzi .
u:Student owl:disjointWith u:Pracownik .
u:Wykladowca owl:equivalentClass [
    a owl:Class ;
    owl:intersectionOf ( u:Pracownik
                         [ a owl:Restriction ; owl:onProperty u:prowadzi ; owl:someValuesFrom u:Kurs ] )
] .
"""
aksjomaty_owl = Graph().parse(data=OWL_TTL, format="turtle")
print("trójek aksjomatów OWL:", len(aksjomaty_owl))

In [ ]:
# Napisz domknij(*grafy, semantyka), która łączy grafy i wywołuje owlrl.DeductiveClosure(semantyka).expand.
# Policz domknięcie RDFS dla graf + schemat: liczba trójek przed i po, ile nowych, ile nowych rdf:type,
# ile trójek rdf:type rdfs:Resource, ile trójek z literałem w podmiocie. Zmierz czas.

In [ ]:
# Jakie typy ma po domknięciu RDFS u:p05 i u:s10? Ilu jest teraz pracowników, studentów i osób?
# Czy ktokolwiek jest u:Wykladowca?

In [ ]:
# Policz domknięcie OWL 2 RL dla graf + schemat + aksjomaty_owl i porównaj z RDFS w jednej tabeli.
# Kto jest teraz u:Wykladowca? Jakie prerekwizyty ma u:ReprezentacjaWiedzy? Kto prowadzi SI (u:prowadzonyPrzez)?

In [ ]:
# Do kopii grafu (graf + schemat + aksjomaty_owl) dodaj trójkę u:s01 u:prowadzi u:Logika.
# Co pokaże domknięcie RDFS (typy u:s01), a co domknięcie OWL 2 RL? owlrl zapisuje wykryte sprzeczności
# jako trójki z predykatem <http://www.daml.org/2002/03/agents/agent-ont#error>.

### Wnioski do zadania 4

(uzupełnij)

## Zadanie 5: OWL w owlready2 i reasoner HermiT

HermiT jest programem w Javie. Poniższa komórka sprawdza, czy Java jest dostępna, i w razie potrzeby ją instaluje (Colab, Debian/Ubuntu).

In [ ]:
if shutil.which("java") is None:
    subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

In [ ]:
DEKLARACJE_TTL = """
@prefix u:    <http://example.org/uczelnia#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .

<http://example.org/uczelnia> a owl:Ontology .
u:Osoba a owl:Class . u:Student a owl:Class . u:Pracownik a owl:Class . u:Wykladowca a owl:Class .
u:Kurs a owl:Class . u:KursObowiazkowy a owl:Class . u:KursObieralny a owl:Class .
u:Wydzial a owl:Class . u:PracaDyplomowa a owl:Class .
u:prowadzi a owl:ObjectProperty . u:zapisanyNa a owl:ObjectProperty . u:zaliczyl a owl:ObjectProperty .
u:wymagaKursu a owl:ObjectProperty . u:naWydziale a owl:ObjectProperty .
u:promotor a owl:ObjectProperty . u:autor a owl:ObjectProperty .
u:imie a owl:DatatypeProperty . u:nazwisko a owl:DatatypeProperty . u:rokStudiow a owl:DatatypeProperty .
u:ects a owl:DatatypeProperty . u:rokUrodzenia a owl:DatatypeProperty .
"""
deklaracje = Graph().parse(data=DEKLARACJE_TTL, format="turtle")

def wczytaj_ontologie(*grafy):
    # Każde wywołanie tworzy osobny świat owlready2, więc eksperymenty sobie nie przeszkadzają
    calosc = Graph()
    for g in grafy:
        calosc += g
    swiat = World()
    onto = swiat.get_ontology("http://example.org/uczelnia").load(
        fileobj=io.BytesIO(calosc.serialize(format="xml").encode("utf-8")))
    return swiat, onto

In [ ]:
# Wczytaj ontologię: swiat, onto = wczytaj_ontologie(graf, schemat, deklaracje)
# Dodaj aksjomaty w owlready2 (najlepiej w funkcji dodaj_aksjomaty(onto), przyda się w 5.3):
#   Wykladowca równoważny Pracownik & prowadzi.some(Kurs); wymagaKursu przechodnia; Student i Pracownik rozłączne
# Uruchom sync_reasoner_hermit(swiat, infer_property_values=True), zmierz czas,
# porównaj instancje Wykladowca z wynikiem OWL 2 RL, wypisz p05.is_a, s10.is_a i prerekwizyty ReprezentacjaWiedzy

In [ ]:
# Zdefiniuj klasę StudentBezZaliczen równoważną Student & zaliczyl.max(0, Kurs) i uruchom HermiT.
# Którzy studenci nie mają w danych żadnego zaliczenia, a kogo reasoner zaliczył do tej klasy?
# Zamknij świat dla u:s08 (close_world(onto.s08)), uruchom ponownie HermiT i porównaj. Co dodało close_world?

In [ ]:
# Na świeżym świecie (wczytaj_ontologie + dodaj_aksjomaty) zdefiniuj klasę StudentPracownik
# będącą podklasą Student i Pracownik; uruchom HermiT i wypisz swiat2.inconsistent_classes().
# Następnie dodaj u:s01 prowadzi u:Logika, uruchom HermiT w try/except OwlReadyInconsistentOntologyError.

### Wnioski do zadania 5

(uzupełnij)

## Zadanie 6: Ontologia z WebProtégé

Scenariusze W1 do W4 z instrukcji wykonujecie w przeglądarce, w WebProtégé. W scenariuszu W4 pobierzcie projekt w formacie RDF/XML albo Turtle. WebProtégé zapisuje pobrany projekt jako archiwum `.zip`. Wgrajcie do Colaba (panel Pliki) samo archiwum albo wypakowany plik pod nazwą `uczelnia_webprotege.owl` lub `uczelnia_webprotege.ttl`.

Jeśli żadnego z tych plików nie ma, komórka wczytująca wypisze komunikat i użyje przykładu przygotowanego w tym samym układzie: pliku `uczelnia_webprotege_przyklad.owl` z materiałów, a gdy i jego brak, tekstu z komórki poniżej. Przykład zawiera stan ontologii po scenariuszach W1 do W3.

In [ ]:
# Przykład w Turtle, ta sama treść co w pliku uczelnia_webprotege_przyklad.owl (RDF/XML)
PRZYKLAD_WEBPROTEGE_TTL = """
@prefix u:    <http://example.org/uczelnia#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .


# ---------- ontologia ----------

<urn:webprotege:ontology:3f6c2a1e-8b7d-4c59-9e0a-5d2b7c41a9f0> a owl:Ontology ;
    rdfs:comment "Plik przykładowy do laboratorium 1. Przygotowany ręcznie w układzie eksportu RDF/XML z WebProtégé (nie jest zapisem rzeczywistej sesji). Odpowiada stanowi po scenariuszach W1 do W3."@pl .


# ---------- właściwości obiektowe ----------

u:autor a owl:ObjectProperty ;
    rdfs:label "author"@en,
        "autor"@pl ;
    rdfs:domain u:PracaDyplomowa ;
    rdfs:range u:Student .

u:naWydziale a owl:ObjectProperty ;
    rdfs:label "in faculty"@en,
        "na wydziale"@pl ;
    rdfs:range u:Wydzial .

u:promotor a owl:ObjectProperty ;
    rdfs:label "supervisor"@en,
        "promotor"@pl ;
    rdfs:domain u:PracaDyplomowa ;
    rdfs:range u:Pracownik .

u:prowadzi a owl:ObjectProperty ;
    rdfs:label "teaches"@en,
        "prowadzi"@pl ;
    rdfs:domain u:Pracownik ;
    rdfs:range u:Kurs ;
    owl:inverseOf u:prowadzonyPrzez .

u:prowadzonyPrzez a owl:ObjectProperty ;
    rdfs:label "taught by"@en,
        "prowadzony przez"@pl .

u:wymagaKursu a owl:ObjectProperty,
        owl:TransitiveProperty ;
    rdfs:label "requires course"@en,
        "wymaga kursu"@pl ;
    rdfs:domain u:Kurs ;
    rdfs:range u:Kurs .

u:zaliczyl a owl:ObjectProperty ;
    rdfs:label "passed"@en,
        "zaliczył"@pl ;
    rdfs:domain u:Student ;
    rdfs:range u:Kurs .

u:zapisanyNa a owl:ObjectProperty ;
    rdfs:label "enrolled in"@en,
        "zapisany na"@pl ;
    rdfs:domain u:Student ;
    rdfs:range u:Kurs .


# ---------- właściwości danych ----------

u:ects a owl:DatatypeProperty,
        owl:FunctionalProperty ;
    rdfs:label "ECTS credits"@en,
        "punkty ECTS"@pl ;
    rdfs:domain u:Kurs ;
    rdfs:range xsd:integer .

u:imie a owl:DatatypeProperty ;
    rdfs:label "first name"@en,
        "imię"@pl ;
    rdfs:domain u:Osoba ;
    rdfs:range xsd:string .

u:nazwisko a owl:DatatypeProperty ;
    rdfs:label "last name"@en,
        "nazwisko"@pl ;
    rdfs:domain u:Osoba ;
    rdfs:range xsd:string .

u:rokStudiow a owl:DatatypeProperty,
        owl:FunctionalProperty ;
    rdfs:label "year of study"@en,
        "rok studiów"@pl ;
    rdfs:domain u:Student ;
    rdfs:range xsd:integer .


# ---------- klasy ----------

u:Kurs a owl:Class ;
    rdfs:label "Course"@en,
        "Kurs"@pl ;
    rdfs:comment "A unit of teaching with ECTS credits."@en,
        "Zajęcia z przypisaną liczbą punktów ECTS."@pl ;
    rdfs:subClassOf [ a owl:Restriction ;
            owl:onClass u:Wydzial ;
            owl:onProperty u:naWydziale ;
            owl:qualifiedCardinality "1"^^xsd:nonNegativeInteger ] .

u:KursObieralny a owl:Class ;
    rdfs:label "Elective course"@en,
        "Kurs obieralny"@pl ;
    rdfs:subClassOf u:Kurs .

u:KursObowiazkowy a owl:Class ;
    rdfs:label "Mandatory course"@en,
        "Kurs obowiązkowy"@pl ;
    rdfs:subClassOf u:Kurs ;
    owl:disjointWith u:KursObieralny .

u:Osoba a owl:Class ;
    rdfs:label "Person"@en,
        "Osoba"@pl ;
    rdfs:comment "A person affiliated with the university."@en,
        "Człowiek związany z uczelnią."@pl .

u:PracaDyplomowa a owl:Class ;
    rdfs:label "Thesis"@en,
        "Praca dyplomowa"@pl ;
    rdfs:subClassOf [ a owl:Restriction ;
            owl:onClass u:Pracownik ;
            owl:onProperty u:promotor ;
            owl:qualifiedCardinality "1"^^xsd:nonNegativeInteger ],
        [ a owl:Restriction ;
            owl:minQualifiedCardinality "1"^^xsd:nonNegativeInteger ;
            owl:onClass u:Student ;
            owl:onProperty u:autor ] .

u:Pracownik a owl:Class ;
    rdfs:label "Employee"@en,
        "Pracownik"@pl ;
    rdfs:comment "A person employed by the university."@en,
        "Osoba zatrudniona na uczelni."@pl ;
    rdfs:subClassOf u:Osoba .

u:Student a owl:Class ;
    rdfs:label "Student"@en,
        "Student"@pl ;
    rdfs:comment "A person studying at the university."@en,
        "Osoba studiująca na uczelni."@pl ;
    rdfs:subClassOf u:Osoba ;
    owl:disjointWith u:Pracownik .

u:Wydzial a owl:Class ;
    rdfs:label "Faculty"@en,
        "Wydział"@pl .

u:Wykladowca a owl:Class ;
    rdfs:label "Lecturer"@en,
        "Wykładowca"@pl ;
    rdfs:comment "An employee who teaches at least one course."@en,
        "Pracownik, który prowadzi co najmniej jeden kurs."@pl ;
    rdfs:subClassOf u:Pracownik ;
    owl:equivalentClass [ a owl:Class ;
            owl:intersectionOf ( u:Pracownik [ a owl:Restriction ;
                        owl:onProperty u:prowadzi ;
                        owl:someValuesFrom u:Kurs ] ) ] .


# ---------- osobniki ----------

u:Algorytmy a u:KursObowiazkowy,
        owl:NamedIndividual ;
    rdfs:label "Algorytmy i struktury danych"@pl ;
    u:ects 6 ;
    u:naWydziale u:WI ;
    u:wymagaKursu u:Programowanie .

u:Logika a u:KursObowiazkowy,
        owl:NamedIndividual ;
    rdfs:label "Logika dla informatyków"@pl ;
    u:ects 4 ;
    u:naWydziale u:WI .

u:Programowanie a u:KursObowiazkowy,
        owl:NamedIndividual ;
    rdfs:label "Programowanie"@pl ;
    u:ects 6 ;
    u:naWydziale u:WI .

u:ReprezentacjaWiedzy a u:KursObieralny,
        owl:NamedIndividual ;
    rdfs:label "Metody reprezentacji wiedzy"@pl ;
    u:ects 4 ;
    u:naWydziale u:WI ;
    u:wymagaKursu u:SztucznaInteligencja .

u:SztucznaInteligencja a u:KursObowiazkowy,
        owl:NamedIndividual ;
    rdfs:label "Sztuczna inteligencja"@pl ;
    u:ects 5 ;
    u:naWydziale u:WI ;
    u:wymagaKursu u:Algorytmy,
        u:Logika .

u:WI a u:Wydzial,
        owl:NamedIndividual ;
    rdfs:label "Wydział Informatyki"@pl .

u:p01 a u:Pracownik,
        owl:NamedIndividual ;
    u:imie "Anna"^^xsd:string ;
    u:naWydziale u:WI ;
    u:nazwisko "Kowalska"^^xsd:string ;
    u:prowadzi u:ReprezentacjaWiedzy,
        u:SztucznaInteligencja .

u:p04 a u:Pracownik,
        owl:NamedIndividual ;
    u:imie "Barbara"^^xsd:string ;
    u:naWydziale u:WI ;
    u:nazwisko "Wójcik"^^xsd:string .

u:p05 a owl:NamedIndividual ;
    u:imie "Tomasz"^^xsd:string ;
    u:nazwisko "Kamiński"^^xsd:string ;
    u:prowadzi u:Logika .

u:pd01 a u:PracaDyplomowa,
        owl:NamedIndividual ;
    rdfs:label "Ontologia planu studiów w OWL"@pl ;
    u:autor u:s01 ;
    u:promotor u:p01 .

u:s01 a u:Student,
        owl:NamedIndividual ;
    u:imie "Jan"^^xsd:string ;
    u:nazwisko "Wiśniewski"^^xsd:string ;
    u:rokStudiow 1 ;
    u:zaliczyl u:Algorytmy,
        u:Logika,
        u:Programowanie,
        u:SztucznaInteligencja ;
    u:zapisanyNa u:ReprezentacjaWiedzy .

u:s04 a u:Student,
        owl:NamedIndividual ;
    u:imie "Agnieszka"^^xsd:string ;
    u:nazwisko "Jankowska"^^xsd:string ;
    u:rokStudiow 1 ;
    u:zaliczyl u:Programowanie ;
    u:zapisanyNa u:ReprezentacjaWiedzy .
"""

In [ ]:
PLIKI_STUDENTA = ["uczelnia_webprotege.owl", "uczelnia_webprotege.ttl"]
PLIK_PRZYKLADU = "uczelnia_webprotege_przyklad.owl"

def znajdz_eksport():
    # Kolejność: plik studenta, archiwum pobrane z WebProtégé, plik przykładowy, przykład z notebooka
    for nazwa in PLIKI_STUDENTA:
        if Path(nazwa).exists():
            return nazwa, Path(nazwa).read_text(encoding="utf-8")
    for archiwum in sorted(Path(".").glob("*ontologies*.zip")):
        with zipfile.ZipFile(archiwum) as z:
            for nazwa in z.namelist():
                if nazwa.endswith((".owl", ".ttl")):
                    return nazwa, z.read(nazwa).decode("utf-8")
    print("Nie znaleziono eksportu z WebProtégé (uczelnia_webprotege.owl, uczelnia_webprotege.ttl ani archiwum .zip).")
    if Path(PLIK_PRZYKLADU).exists():
        print("Używam pliku przykładowego", PLIK_PRZYKLADU)
        return PLIK_PRZYKLADU, Path(PLIK_PRZYKLADU).read_text(encoding="utf-8")
    print("Używam przykładu z komórki PRZYKLAD_WEBPROTEGE_TTL")
    return "przyklad.ttl", PRZYKLAD_WEBPROTEGE_TTL

def swiat_z_eksportu(g):
    # Eksport z WebProtégé ma własne IRI ontologii (domyślnie urn:webprotege:ontology:...). Gdyby wczytać go
    # przez wczytaj_ontologie, owlready2 utworzyłby pustą ontologię o IRI równym przestrzeni u: i wnioski
    # o wartościach właściwości trafiałyby w złe miejsce. Dlatego osobna, neutralna nazwa.
    swiat = World()
    onto = swiat.get_ontology("http://example.org/eksport-webprotege").load(
        fileobj=io.BytesIO(g.serialize(format="xml").encode("utf-8")))
    return swiat, onto

def encja(swiat, nazwa):
    # Wyszukanie po pełnym IRI; przy IRI z losowym identyfikatorem encji u:nazwa nie będzie
    e = swiat[str(U[nazwa])]
    if e is None:
        print(f"brak encji u:{nazwa}, sprawdźcie IRI nadawane w ustawieniach projektu")
    return e

zrodlo_wp, tekst_wp = znajdz_eksport()
wp = Graph().parse(data=tekst_wp, format="turtle" if zrodlo_wp.endswith(".ttl") else "xml")
wp.bind("u", U)
print("źródło:", zrodlo_wp, "| trójek:", len(wp))

In [ ]:
# 6.1 Gotowa komórka. Co jest w eksporcie
def nazwane(typ):
    return sorted({s for s in wp.subjects(RDF.type, typ) if isinstance(s, URIRef)}, key=str)

rodzaje = {"klasy": OWL.Class, "właściwości obiektowe": OWL.ObjectProperty,
           "właściwości danych": OWL.DatatypeProperty, "osobniki": OWL.NamedIndividual}
print(pd.Series({k: len(nazwane(t)) for k, t in rodzaje.items()}, name="liczba").to_string())
obce = [str(e) for t in rodzaje.values() for e in nazwane(t) if not str(e).startswith(str(U))]
print("encje spoza przestrzeni u:", obce if obce else "brak")

def etykieta(zasob, jezyk):
    return next((str(l) for l in wp.objects(zasob, RDFS.label) if l.language == jezyk), None)

klasy_wp = pd.DataFrame([{"klasa": krotko(k), "pl": etykieta(k, "pl"), "en": etykieta(k, "en"),
                          "nadklasa": ", ".join(krotko(n) for n in wp.objects(k, RDFS.subClassOf) if isinstance(n, URIRef)),
                          "komentarze": len(list(wp.objects(k, RDFS.comment)))} for k in nazwane(OWL.Class)])
klasy_wp

In [ ]:
# 6.2 Gotowa komórka. Aksjomaty zapisane w WebProtégé
display(tabela(wp, '''
SELECT ?a ?aksjomat ?b WHERE {
  { ?a a ?aksjomat . FILTER(?aksjomat IN (owl:TransitiveProperty, owl:FunctionalProperty)) }
  UNION { ?a owl:inverseOf ?b . BIND(owl:inverseOf AS ?aksjomat) }
  UNION { ?a owl:disjointWith ?b . BIND(owl:disjointWith AS ?aksjomat) }
  UNION { ?a owl:equivalentClass ?x . BIND(owl:equivalentClass AS ?aksjomat)
          BIND(IF(isBlank(?x), "wyrażenie klasowe", ?x) AS ?b) }
} ORDER BY ?aksjomat ?a'''))
tabela(wp, '''
SELECT ?klasa ?wlasciwosc ?ograniczenie ?n ?klasa_wartosci WHERE {
  ?klasa rdfs:subClassOf ?r .
  ?r a owl:Restriction ; owl:onProperty ?wlasciwosc ; ?ograniczenie ?n .
  FILTER(?ograniczenie IN (owl:cardinality, owl:minCardinality, owl:maxCardinality,
                           owl:qualifiedCardinality, owl:minQualifiedCardinality, owl:maxQualifiedCardinality))
  OPTIONAL { ?r owl:onClass ?klasa_wartosci }
} ORDER BY ?klasa''')

In [ ]:
# 6.3 Wczytaj wp do nowego świata owlready2 funkcją swiat_z_eksportu(wp) (deklaracje są już w eksporcie) i uruchom HermiT.
# Wypisz klasy niespełnialne, wykładowców zapisanych wprost i wywnioskowanych, p05.is_a,
# prerekwizyty ReprezentacjaWiedzy i prowadzących SztucznaInteligencja (prowadzonyPrzez).
# Encje wyszukuj funkcją encja(swiat, "Nazwa"), np. encja(swiat_wp, "Wykladowca").

In [ ]:
# 6.4 Na świeżym świecie dodaj osobnika x01 będącego jednocześnie Student i Pracownik, uruchom HermiT
# w try/except OwlReadyInconsistentOntologyError. Na kolejnym świeżym świecie dodaj pracy pd01 drugiego
# promotora p04. Czy ontologia jest niespójna? Co się zmienia po AllDifferent([p01, p04])?

### Wnioski do zadania 6

(uzupełnij)

## Zadanie 7: Wikidata

Funkcja `zapytaj_wikidata` wysyła zapytanie do publicznego punktu dostępowego, a gdy ten nie odpowiada (awaria, limit zapytań), zwraca dane zapasowe z poniższej komórki. Kolumna `zrodlo` w wyniku mówi, skąd pochodzą dane. Identyfikatory encji są skracane do postaci `Q...`.

In [ ]:
ZAPAS_UCZELNIE = """
uczelnia,nazwa,miasto,rok
Q62391165,Q62391165,Gdańsk,
Q114957293,University of Economics and Human Sciences in Warsaw,Warszawa,
Q189441,Uniwersytet Jagielloński w Krakowie,Kraków,1364
Q315658,Uniwersytet Wrocławski,Wrocław,1702
Q56337486,Królewski Uniwersytet Warszawski,Warszawa,1816
Q1718656,Politechnika Warszawska,Warszawa,1899
Q348402,Uniwersytet im. Adama Mickiewicza w Poznaniu,Poznań,1919
Q7895397,Uniwersytet Artystyczny im. Magdaleny Abakanowicz w Poznaniu,Poznań,1919
Q11684710,Akademia Marynarki Wojennej im. Bohaterów Westerplatte,Gdynia,1922
Q1471879,Uniwersytet Ekonomiczny w Krakowie,Kraków,1924
Q413829,Akademia Wychowania Fizycznego Józefa Piłsudskiego w Warszawie,Bielany,1929
Q137503,Uniwersytet Łódzki,Łódź,1945
Q931642,Politechnika Krakowska im. Tadeusza Kościuszki,Kraków,1945
Q1752712,Uniwersytet Mikołaja Kopernika w Toruniu,Toruń,1945
Q677662,Uniwersytet Komisji Edukacji Narodowej w Krakowie,Kraków,1946
Q11684721,Akademia Wychowania Fizycznego we Wrocławiu,Wrocław,1946
Q2399276,Politechnika Częstochowska,Częstochowa,1949
Q1916610,Uniwersytet Medyczny w Lublinie,Lublin,1950
Q3207844,Uniwersytet Radomski,Radom,1950
Q677503,Politechnika Bydgoska im. Jana i Jędrzeja Śniadeckich,Bydgoszcz,1951
Q2364321,Uniwersytet Przyrodniczy w Poznaniu,Poznań,1951
Q3578303,Politechnika Rzeszowska im. Ignacego Łukasiewicza,Rzeszów,1951
Q4121030,Uniwersytet Przyrodniczy we Wrocławiu,Wrocław,1951
Q1804769,Uniwersytet Rolniczy im. Hugona Kołłątaja w Krakowie,Kraków,1953
Q5734894,Uniwersytet Przyrodniczy w Lublinie,Lublin,1955
Q615154,Uniwersytet Śląski w Katowicach,Katowice,1968
Q1737513,Uniwersytet Kazimierza Wielkiego w Bydgoszczy,Bydgoszcz,1969
Q3551420,Uniwersytet Jana Kochanowskiego w Kielcach,Kielce,1969
Q9366819,Uniwersytet w Siedlcach,Siedlce,1969
Q1758009,Uniwersytet Gdański,Gdańsk,1970
Q5180437,Krakowskie Szkoły Artystyczne,Kraków,1989
Q1658883,Uniwersytet w Białymstoku,Białystok,1997
Q8040714,Wyższa Szkoła Menedżerska w Legnicy,Legnica,1997
Q11798134,Olsztyńska Szkoła Wyższa,Olsztyn,1997
Q9380716,Wyższa Szkoła Bankowa w Toruniu,Toruń,1998
Q1733433,Uniwersytet Warmińsko-Mazurski w Olsztynie,Olsztyn,1999
Q7895365,Wyższa Szkoła Ekonomii i Innowacji w Lublinie,Lublin,2000
Q366071,Uniwersytet Zielonogórski,Zielona Góra,2001
Q614293,Międzynarodowa Wyższa Szkoła Logistyki i Transportu we Wrocławiu,Wrocław,2001
Q2399398,Uniwersytet Bielsko-Bialski,Bielsko-Biała,2001
"""

ZAPAS_LUDNOSC = """
liczba,data
1862402,2024-06-30T00:00:00Z
1860281,2021-03-31T00:00:00Z
1790658,2019-12-31T00:00:00Z
1777972,2018-12-31T00:00:00Z
1764615,2017-12-31T00:00:00Z
1753977,2016-12-31T00:00:00Z
1744351,2015-12-31T00:00:00Z
1735442,2014-12-31T00:00:00Z
"""

In [ ]:
WIKIDATA = "https://query.wikidata.org/sparql"
NAGLOWKI = {"User-Agent": "MRW-lab/1.0 (zajecia dydaktyczne)", "Accept": "application/sparql-results+json"}

def zapytaj_wikidata(zapytanie, zapas, proby=2):
    blad = None
    for proba in range(proby):
        try:
            r = requests.get(WIKIDATA, params={"query": zapytanie}, headers=NAGLOWKI, timeout=60)
            if r.status_code == 429 and proba + 1 < proby:
                naglowek = r.headers.get("Retry-After", "30")
                time.sleep(min(int(naglowek) if naglowek.isdigit() else 30, 70))
                continue
            r.raise_for_status()
            wiersze = [{k: v["value"] for k, v in b.items()} for b in r.json()["results"]["bindings"]]
            df = pd.DataFrame(wiersze)
            for c in df.columns:
                df[c] = df[c].str.replace("http://www.wikidata.org/entity/", "", regex=False)
            df["zrodlo"] = "wikidata"
            return df
        except Exception as e:
            blad = e
    print("Wikidata nie odpowiada (", type(blad).__name__, "), używam danych zapasowych")
    df = pd.read_csv(io.StringIO(zapas), dtype=str, keep_default_na=False)
    df["zrodlo"] = "zapas"
    return df

In [ ]:
Q_UCZELNIE = """
SELECT ?uczelnia ?nazwa ?miasto ?rok WHERE {
  ?uczelnia wdt:P31 wd:Q3918 ; wdt:P17 wd:Q36 .
  OPTIONAL { ?uczelnia wdt:P131 ?m . }
  OPTIONAL { ?uczelnia wdt:P571 ?zal . BIND(YEAR(?zal) AS ?rok) }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en" .
    ?uczelnia rdfs:label ?nazwa . ?m rdfs:label ?miasto . }
}
ORDER BY ?rok
LIMIT 40"""
# Wykonaj zapytanie przez zapytaj_wikidata(Q_UCZELNIE, ZAPAS_UCZELNIE), zamień kolumnę rok na liczby.
# Ile wierszy nie ma roku założenia, ile nie ma nazwy (nazwa równa identyfikatorowi), czy encje się powtarzają?

In [ ]:
Q_LUDNOSC = """
SELECT ?liczba ?data WHERE {
  wd:Q270 p:P1082 ?st .
  ?st ps:P1082 ?liczba .
  OPTIONAL { ?st pq:P585 ?data . }
}
ORDER BY DESC(?data)
LIMIT 8"""
# Wykonaj Q_LUDNOSC (zapas: ZAPAS_LUDNOSC). Potem zapytaj o to samo przez wdt:P1082 (bez p:/ps:/pq:)
# i porównaj liczbę zwróconych wartości.

### Wnioski do zadania 7

(uzupełnij)

## Zadanie 8: Pytania kontrolne

1. Czym różni się dziedziczenie w sieci semantycznej z wyjątkami od wnioskowania w RDFS i dlaczego to pierwsze jest niemonotoniczne?
2. Dlaczego zapytanie o `?p a u:Pracownik` nie zwraca u:p05 przed wnioskowaniem, a zwraca po domknięciu RDFS?
3. Czego nie da się wyrazić w RDFS, a da się w OWL? Podajcie dwa przykłady z tego laboratorium.
4. Dlaczego HermiT nie zaliczył nikogo do klasy StudentBezZaliczen, choć część studentów nie ma w danych żadnego zaliczenia, a SPARQL z `FILTER NOT EXISTS` takich studentów znajduje?
5. Czym różni się klasa niespełnialna od niespójnej ontologii?
6. Czym różni się `wdt:P31` od `rdf:type`, a `wdt:P279` od `rdfs:subClassOf`?

Odpowiedzi (uzupełnij):
7. Dlaczego w WebProtégé u:p05 nie jest wykładowcą, a w notebooku po eksporcie jest? Co z tego wynika dla podziału pracy między edytorem a reasonerem?